# UPS vs FedEx: market data, risk and machine-learning price forecasting

This notebook is the data-science half of the project. It:

1. Downloads **~16 years of daily prices** for UPS, FedEx, the S&P 500 (SPY) and the transport sector (IYT).
2. Measures **risk and return**: annual total returns, volatility, drawdowns, Sharpe ratio and **beta** (the input to WACC).
3. Builds **40+ features** per day and trains four forecasting models on the same data:
   **Naive (random walk)**, **ARIMA**, **Gradient Boosting** and an **LSTM neural network**.
4. Tests every model on the **last 252 trading days** it has never seen, with RMSE, MAE, MAPE and directional accuracy.
5. Simulates **10,000 Monte Carlo paths** for a 30-day forward price range.
6. Saves everything to `outputs/`, which the Streamlit dashboard reads.

**How to run (Google Colab):** `Runtime > Run all`. It takes about 5-10 minutes. At the end a file called
`forecast_outputs.zip` downloads; unzip it into the `outputs/` folder of your GitHub repo.

## Step 0 - Setup

In [ ]:
# Colab already has most of these. This installs anything that is missing.
import importlib, subprocess, sys
for pkg, mod in [("yfinance", "yfinance"), ("statsmodels", "statsmodels"), ("scikit-learn", "sklearn"), ("tensorflow", "tensorflow")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, random, warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

TICKERS = ["UPS", "FDX"]          # companies we forecast
BENCH = ["SPY", "IYT"]            # S&P 500 ETF and iShares Transportation ETF
START = "2010-01-01"
TEST_DAYS = 252                   # the last trading year is held out for testing
WINDOW = 60                       # days of history the LSTM looks at
RISK_FREE = 0.04                  # for the Sharpe ratio

# DEMO_MODE is only for testing the code offline with simulated prices. Leave it off.
DEMO_MODE = os.environ.get("DEMO_MODE") == "1"
OUT = "outputs_demo" if DEMO_MODE else "outputs"
os.makedirs(OUT, exist_ok=True)
print("Saving results to:", OUT)

## Step 1 - Download daily prices

`auto_adjust=True` gives prices adjusted for **dividends** and for the **June 1, 2026 FedEx Freight spin-off**,
so returns are true total returns and the spin-off does not look like a crash.

In [ ]:
def download_prices():
    import yfinance as yf
    data = yf.download(TICKERS + BENCH, start=START, auto_adjust=True, progress=False)
    if data.empty:
        raise RuntimeError("No data downloaded. Check your internet connection and try again.")
    return data["Close"], data["Volume"]


def demo_prices():
    """Simulated prices so the code can be tested without internet. Never used for real results."""
    rng = np.random.default_rng(SEED)
    idx = pd.bdate_range(START, "2026-11-13")
    close, vol = {}, {}
    for t, (mu, sig, p0) in {"UPS": (0.05, 0.27, 60), "FDX": (0.07, 0.33, 110), "SPY": (0.10, 0.17, 110), "IYT": (0.08, 0.22, 85)}.items():
        r = rng.normal(mu / 252, sig / np.sqrt(252), len(idx))
        close[t] = p0 * np.exp(np.cumsum(r))
        vol[t] = rng.lognormal(15, 0.3, len(idx))
    return pd.DataFrame(close, idx), pd.DataFrame(vol, idx)


close, volume = demo_prices() if DEMO_MODE else download_prices()
close = close[TICKERS + BENCH].ffill().dropna()
volume = volume[TICKERS + BENCH].reindex(close.index).ffill()
close.index = pd.to_datetime(close.index).tz_localize(None)
volume.index = close.index
close.round(4).to_csv(f"{OUT}/prices.csv")
print(f"{len(close):,} trading days x {close.shape[1]} tickers, {close.index[0].date()} to {close.index[-1].date()}")
close.tail()

## Step 2 - Risk and return

In [ ]:
rets = close.pct_change().dropna()

# Calendar-year total returns (the latest year is year-to-date)
annual = close.resample("YE").last().pct_change().dropna()
annual.index = annual.index.year
annual.round(4).to_csv(f"{OUT}/annual_returns.csv")

def max_drawdown(s):
    return float((s / s.cummax() - 1).min())

monthly = close.resample("ME").last().pct_change().dropna()
last5 = monthly.iloc[-60:]
stats = []
for t in TICKERS + BENCH:
    ann_ret = (close[t].iloc[-1] / close[t].iloc[-1 - 252 * 5]) ** (1 / 5) - 1
    ann_vol = rets[t].iloc[-252 * 5:].std() * np.sqrt(252)
    beta = last5[t].cov(last5["SPY"]) / last5["SPY"].var()
    stats.append({"Ticker": t, "5y annual return": ann_ret, "5y annual volatility": ann_vol,
                  "Sharpe ratio (5y)": (ann_ret - RISK_FREE) / ann_vol, "Max drawdown (5y)": max_drawdown(close[t].iloc[-252 * 5:]),
                  "Beta vs S&P 500 (5y monthly)": beta, "Correlation with SPY (5y daily)": rets[t].iloc[-252 * 5:].corr(rets["SPY"].iloc[-252 * 5:])})
risk = pd.DataFrame(stats).set_index("Ticker")
risk.round(4).to_csv(f"{OUT}/risk_stats.csv")

# Rolling 5-year beta: how stable is the beta we use in WACC?
roll_beta = pd.DataFrame({t: monthly[t].rolling(60).cov(monthly["SPY"]) / monthly["SPY"].rolling(60).var() for t in TICKERS}).dropna()
roll_beta.round(4).to_csv(f"{OUT}/rolling_beta.csv")

growth = (1000 * close.loc[close.index >= "2016-01-01"] / close.loc[close.index >= "2016-01-01"].iloc[0])
growth.plot(figsize=(11, 4), title="Growth of $1,000 invested in 2016 (dividends reinvested)")
plt.ylabel("$"); plt.show()
risk.style.format("{:.2f}")

## Step 3 - Feature engineering

For each trading day we build features using **only information available that day**, and the target is the
**next day's log return**. Using returns instead of raw prices keeps the models from simply copying yesterday's price.

In [ ]:
def rsi(series, n=14):
    delta = series.diff()
    up = delta.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    down = (-delta.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + up / down)


def make_features(t):
    p = close[t]
    lr = np.log(p).diff()
    f = pd.DataFrame(index=p.index)
    for k in (1, 2, 3, 5, 10, 21):
        f[f"ret_lag{k}"] = lr.shift(k - 1)
    for k in (5, 10, 21, 63):
        f[f"vol_{k}"] = lr.rolling(k).std()
        f[f"mom_{k}"] = np.log(p / p.shift(k))
    for k in (20, 50, 200):
        f[f"dist_ma{k}"] = p / p.rolling(k).mean() - 1
    f["rsi_14"] = rsi(p)
    f["vol_z_21"] = (volume[t] - volume[t].rolling(21).mean()) / volume[t].rolling(21).std()
    for b in BENCH:
        blr = np.log(close[b]).diff()
        for k in (1, 2, 5):
            f[f"{b}_ret_lag{k}"] = blr.shift(k - 1)
        f[f"{b}_vol_21"] = blr.rolling(21).std()
    other = [x for x in TICKERS if x != t][0]
    olr = np.log(close[other]).diff()
    f["peer_ret_lag1"] = olr
    f["peer_mom_21"] = np.log(close[other] / close[other].shift(21))
    f["day_of_week"] = p.index.dayofweek
    f["month"] = p.index.month
    f["target"] = lr.shift(-1)          # tomorrow's log return
    f["price"] = p
    f["next_price"] = p.shift(-1)
    return f.dropna()


FEATS = {t: make_features(t) for t in TICKERS}
FEATURE_COLS = [c for c in FEATS["UPS"].columns if c not in ("target", "price", "next_price")]
print(f"{len(FEATURE_COLS)} features x {len(FEATS['UPS']):,} days per company")
FEATS["UPS"][FEATURE_COLS].tail(3)

## Step 4 - Train and test four models

Every model predicts **tomorrow's price** for each of the last 252 trading days, using only data up to today
(walk-forward, no peeking). The **Naive** model just says "tomorrow = today"; any useful model must beat it.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
import tensorflow as tf

tf.random.set_seed(SEED)
EPOCHS = 2 if DEMO_MODE else 30


def split(f):
    return f.iloc[:-TEST_DAYS], f.iloc[-TEST_DAYS:]


def naive_model(f):
    train, test = split(f)
    return test["price"].values  # tomorrow = today


def arima_model(f):
    train, test = split(f)
    y_train = np.log(train["next_price"].values)
    best = None
    for order in [(1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 2)]:
        try:
            res = ARIMA(y_train, order=order).fit()
            if best is None or res.aic < best[1].aic:
                best = (order, res)
        except Exception:
            continue
    order, res = best
    full = np.log(np.concatenate([train["next_price"].values, test["next_price"].values]))
    applied = res.apply(full)                       # same parameters, filtered through the test period
    pred_log = applied.predict(start=len(y_train), end=len(full) - 1)   # one-step-ahead forecasts
    print(f"  ARIMA order chosen by AIC: {order}")
    return np.exp(pred_log)


def gbm_model(f):
    train, test = split(f)
    model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, max_leaf_nodes=15,
                                          l2_regularization=1.0, random_state=SEED)
    model.fit(train[FEATURE_COLS], train["target"])
    return test["price"].values * np.exp(model.predict(test[FEATURE_COLS]))


def lstm_model(f):
    train, test = split(f)
    scaler_x = StandardScaler().fit(train[FEATURE_COLS])
    X = scaler_x.transform(f[FEATURE_COLS])
    y_scale = train["target"].std()
    y = f["target"].values / y_scale
    n_train = len(train)
    seq = lambda idx: np.stack([X[i - WINDOW + 1:i + 1] for i in idx])  # noqa: E731
    tr_idx = np.arange(WINDOW - 1, n_train)
    te_idx = np.arange(n_train, len(f))
    model = tf.keras.Sequential([
        tf.keras.layers.Input((WINDOW, len(FEATURE_COLS))),
        tf.keras.layers.LSTM(64),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    model.fit(seq(tr_idx), y[tr_idx], epochs=EPOCHS, batch_size=64, validation_split=0.1, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    pred = model.predict(seq(te_idx), verbose=0).ravel() * y_scale
    return test["price"].values * np.exp(pred)


MODELS = {"Naive": naive_model, "ARIMA": arima_model, "Gradient Boosting": gbm_model, "LSTM": lstm_model}
frames, metrics = [], []
for t in TICKERS:
    f = FEATS[t]
    train, test = split(f)
    next_dates = close.index[close.index.get_indexer(test.index) + 1]   # the day each prediction is for
    out = pd.DataFrame({"date": next_dates, "ticker": t, "today": test["price"].values, "actual": test["next_price"].values})
    print(f"{t}: training on {len(train):,} days, testing on {len(test)} days")
    for name, fn in MODELS.items():
        out[name] = fn(f)
    for name in MODELS:
        err = out[name] - out["actual"]
        direction = np.nan if name == "Naive" else float((np.sign(out[name] - out["today"]) == np.sign(out["actual"] - out["today"])).mean())
        metrics.append({"ticker": t, "model": name, "RMSE": float(np.sqrt((err ** 2).mean())), "MAE": float(err.abs().mean()),
                        "MAPE": float((err.abs() / out["actual"]).mean()), "Directional accuracy": direction})
    frames.append(out)

avp = pd.concat(frames, ignore_index=True)
avp.round(4).to_csv(f"{OUT}/actual_vs_predicted.csv", index=False)
m = pd.DataFrame(metrics)
m["RMSE vs naive"] = m["RMSE"] / m.groupby("ticker")["RMSE"].transform(lambda s: s.iloc[0]) - 1
m.round(5).to_csv(f"{OUT}/model_metrics.csv", index=False)
m.style.format({"RMSE": "{:.3f}", "MAE": "{:.3f}", "MAPE": "{:.2%}", "Directional accuracy": "{:.1%}", "RMSE vs naive": "{:+.2%}"})

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=False)
for ax, t in zip(axes, TICKERS):
    d = avp[avp.ticker == t].set_index("date")
    d[["actual", "ARIMA", "Gradient Boosting", "LSTM"]].plot(ax=ax, lw=1.2)
    ax.set_title(f"{t}: actual vs predicted next-day price (test year)")
plt.tight_layout(); plt.show()

## Step 5 - 30-day forward range (Monte Carlo)

Point forecasts of a stock price a month out are unreliable, so we show a **range**. We draw 10,000 paths by
re-sampling the last two years of real daily returns (a *bootstrap*), which keeps fat tails and volatility realistic.

In [ ]:
HORIZON, N_PATHS = 30, 10_000
rng = np.random.default_rng(SEED)
rows = []
for t in TICKERS:
    hist = np.log(close[t]).diff().dropna().iloc[-504:].values
    draws = rng.choice(hist, size=(N_PATHS, HORIZON), replace=True)
    paths = close[t].iloc[-1] * np.exp(np.cumsum(draws, axis=1))
    dates = pd.bdate_range(close.index[-1] + pd.offsets.BDay(1), periods=HORIZON)
    pct = np.percentile(paths, [5, 25, 50, 75, 95], axis=0)
    for i, d in enumerate(dates):
        rows.append({"date": d, "ticker": t, "day": i + 1, "last_price": close[t].iloc[-1],
                     "p5": pct[0, i], "p25": pct[1, i], "p50": pct[2, i], "p75": pct[3, i], "p95": pct[4, i]})
fc = pd.DataFrame(rows)
fc.round(4).to_csv(f"{OUT}/forecast_30d.csv", index=False)
fc[fc.day == HORIZON]

## Step 6 - Save and download

In [ ]:
meta = {"run_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "data_start": str(close.index[0].date()),
        "data_end": str(close.index[-1].date()), "test_days": TEST_DAYS, "demo_mode": DEMO_MODE,
        "tickers": TICKERS + BENCH, "n_features": len(FEATURE_COLS), "monte_carlo_paths": N_PATHS}
json.dump(meta, open(f"{OUT}/run_metadata.json", "w"), indent=2)

import shutil
shutil.make_archive("forecast_outputs", "zip", OUT)
try:
    from google.colab import files
    files.download("forecast_outputs.zip")
    print("Downloading forecast_outputs.zip - unzip it into the outputs/ folder of your repo.")
except ImportError:
    print(f"Done. Results are in the '{OUT}' folder.")

## What the results mean

* Daily stock prices are **very close to a random walk**, so the Naive model is hard to beat. If ARIMA, Gradient Boosting
  or the LSTM beat it by even a few percent on RMSE, or call the direction right more than ~52% of the time, that is a
  meaningful result. If they do not, that is a finding too: it is why the **valuation** in this project rests on
  cash-flow fundamentals (the DCF), not on price patterns.
* **Beta** from Step 2 feeds the cost of equity in the WACC. Compare it with the beta on the Assumptions sheet of the Excel model.
* This is an educational project, not investment advice.